# 04 · Model return: varian A–D, kontrol, ablasi (hanya Train dan Val)
A harga · B +GPR/sentimen · C +embedding/topik/fitur time series · D +`p_up`. Ridge, TimeSeriesSplit(5, gap 5), aturan 1-SE.

In [1]:
import json
import numpy as np, pandas as pd
from utils import *

df, price = load()
gate = json.load(open(PROC / 'gate.json'))
cols = [c for c in df.columns if c not in ('target', 'target2', 'target0', 'split')]
dv = df[df.split.isin(['train', 'val'])]
base = dv[dv.p_up.notna()]                   # semua varian memakai baris yang sama (fold pertama OOF dibuang)
tr, va = base[base.split == 'train'], base[base.split == 'val']
y, V = va.target.values, variants(cols, price)
assert not base[cols].isna().any().any()
print('train', len(tr), '| val', len(va), '| gate classifier:', gate['gate'])

train 80 | val 21 | gate classifier: True


In [2]:
def run(name, trf=tr, vaf=va, ycol='target'):
    keep, use = V[name]
    return fit_predict(trf[use], trf[ycol], vaf[use], keep)

P = {n: run(n) for n in V}
maj = 1 if (tr.target > 0).mean() > .5 else -1
rows = [{'model': n, **evaluate(y, p, tr.target.std())} for n, p in P.items()]
rows += [{'model': 'zero', **evaluate(y, 0 * y)}, {'model': 'kemarin', **evaluate(y, va.ret_lag0.values)},
         {'model': 'mayoritas', 'DirAcc': np.mean(np.sign(y) == maj)}]
res = pd.DataFrame(rows).set_index('model')
res['DM_vs_A_p'] = [dm_test(y, P['A'], P[n])[1] if n in 'BCD' else np.nan for n in res.index]
print(res.round(5).to_string())

              RMSE      MAE   DirAcc collapsed  DM_vs_A_p
model                                                    
A          0.00269  0.00224  0.57143      True        NaN
B          0.00269  0.00224  0.57143      True    0.53812
C          0.00269  0.00224  0.57143      True    0.61249
D          0.00269  0.00224  0.57143      True    0.63297
zero       0.00275  0.00231  0.04762       NaN        NaN
kemarin    0.00409  0.00348  0.42857       NaN        NaN
mayoritas      NaN      NaN  0.57143       NaN        NaN


## Kontrol negatif: acak urutan hari fitur teks (hanya Train), latih ulang
Untuk D, classifier juga dilatih ulang pada fitur teks yang diacak.

In [3]:
textc = [c for c in V['C'][1] if c not in price]
rm = lambda p: np.sqrt(np.mean((y - p) ** 2))
neg = {'C': [], 'D': []}
for s in range(20):
    sh = dv.copy()
    sh.loc[sh.split == 'train', textc] = sh.loc[sh.split == 'train', textc].sample(frac=1, random_state=s).values
    neg['C'].append(rm(run('C', sh.loc[tr.index], va)))
    sh['p_up'] = oof_pup(sh.drop(columns='p_up'), textc)
    sh = sh.loc[base.index]
    neg['D'].append(rm(run('D', sh[sh.split == 'train'], sh[sh.split == 'val'])))
for n in neg: print(f'{n}: asli {rm(P[n]):.5f} | acak {np.mean(neg[n]):.5f} ± {np.std(neg[n]):.5f} (min {np.min(neg[n]):.5f})')
print('Kalau "asli" tidak jelas lebih kecil dari distribusi acak, keuntungan berasal dari shrinkage, bukan berita.')

C: asli 0.00269 | acak 0.00269 ± 0.00000 (min 0.00268)
D: asli 0.00269 | acak 0.00269 ± 0.00000 (min 0.00268)
Kalau "asli" tidak jelas lebih kecil dari distribusi acak, keuntungan berasal dari shrinkage, bukan berita.


## Target digeser dan ablasi
Pergeseran memakai fitur teks saja (fitur harga memuat `ret[t]`, jadi `target0` jadi trivial).

In [4]:
sh_rows = []
for col in ('target0', 'target', 'target2'):
    a, b = tr.dropna(subset=[col]), va.dropna(subset=[col])
    p = fit_predict(a[textc], a[col], b[textc], [])
    sh_rows.append({'target': col, 'RMSE/RMSE_zero': np.sqrt(np.mean((b[col] - p) ** 2)) / np.sqrt(np.mean(b[col] ** 2)), 'corr': np.corrcoef(p, b[col])[0, 1]})
print(pd.DataFrame(sh_rows).round(3).to_string(index=False))
print('target0 (ret[t]) berkorelasi tinggi = fitur memuat info hari yang sama. target2 lebih baik dari target = curiga salah geser.')

keepD, useD = V['D']
abl = {}
for b in ['cnt', 'gpr', 'sent', 'emb', 'topic', 'pup']:
    p = fit_predict(tr[[c for c in useD if block(c) != b]], tr.target, va[[c for c in useD if block(c) != b]], [c for c in keepD if block(c) != b])
    abl[b] = rm(p) - rm(P['D'])
print('\nAblasi (Δ RMSE Val terhadap D, positif = blok itu membantu):'); print(pd.Series(abl).round(6).to_string())

 target  RMSE/RMSE_zero  corr
target0           1.031 0.081
 target           0.975 0.128
target2           0.986 0.022
target0 (ret[t]) berkorelasi tinggi = fitur memuat info hari yang sama. target2 lebih baik dari target = curiga salah geser.

Ablasi (Δ RMSE Val terhadap D, positif = blok itu membantu):
cnt      0.000000
gpr      0.000001
sent     0.000000
emb      0.000001
topic    0.000000
pup      0.000000


In [5]:
best = res.loc[list(V)].RMSE.idxmin()
json.dump(dict(finalist=best), open(PROC / 'finalist.json', 'w'))
print('finalis (RMSE Val terkecil):', best, '| zero:', round(res.loc['zero', 'RMSE'], 5))

finalis (RMSE Val terkecil): D | zero: 0.00275
